# Isoprene Sensitivity Studies

---
Last modified: February 6th, 2025 (JY)

This notebook uses a modified version of vSmartMOM with functions specifically for isoprene to calculate radiances with different amounts of isoprene/water vapor/aerosols/etc.

This notebook is coded in Julia.

In [2]:
# Activate Julia packages
using Pkg;
using Revise;

Pkg.activate("vSmartMOM_ISOP.jl");
Pkg.instantiate();

# Using local module, since the current version of vSmartMOM does not support isoprene (not a HITRAN line list species)
include("vSmartMOM_ISOP.jl/src/vSmartMOM.jl");

# Other imports
using Statistics, PlutoUI, Polynomials, PlotlyBase, Plots, LazyArtifacts, LinearAlgebra, NCDatasets, Format, LaTeXStrings, .vSmartMOM;
include("helper_functions.jl");

default(fontfamily="Helvetica");

  Activating project at `~/Documents/Radiative_Transfer/vSmartMOM_ISOP.jl`


### We now need to specify parameters for the model.

I have downloaded 10 days (2019-07-01 to 2019-07-10) in the MERRA2 folder. You will need to manually download more days if you would like different conditions.

In [3]:
function retrieve_merra2_conditions(date, startTime, lat, lon)
    # Feel free to change the MERRA2 file and folder paths
    MerraFile = "MERRA2_400.inst6_3d_ana_Nv." * date * ".nc4"
    MerraFolder = "MERRA2/"
    MerraFilePath = joinpath(MerraFolder, MerraFile)

    # These files have 00, 06, 12 or 18 in UTC, i.e. 6 hourly data stacked together
    hour = parse(Int, chop(startTime));
    hour_index = Int(hour / 6);

    # Read atmos profile (from helperfunctions.jl)
    return read_atmos_profile(MerraFilePath, lat, lon, hour_index);
end

# Parameters for atmospheric profile
date = "20190701"
startTime = "06z"
lat = 2.
lon = 100.

merra2_profile = retrieve_merra2_conditions(date, startTime, lat, lon);

pressures = merra2_profile.p_levels ./ 100; # Convert to hPa
pressures_midpoints = merra2_profile.p ./ 100; # Convert to hPa
temperatures = merra2_profile.T;
specific_humidity = merra2_profile.q .* 1000; # Convert from kg/kg (in MERRA2) to g/kg;

ds["T"] = T (576 × 361 × 72 × 4)
  Datatype:    Union{Missing, Float32} (Float32)
  Dimensions:  lon × lat × lev × time
  Attributes:
   long_name            = Air temperature
   units                = K
   _FillValue           = 1.0e15
   missing_value        = 1.0e15
   fmissing_value       = 1.0e15
   scale_factor         = 1.0
   add_offset           = 0.0
   standard_name        = air_temperature
   vmax                 = 1.0e15
   vmin                 = -1.0e15
   valid_range          = Float32[-1.0f15, 1.0f15]



In [45]:
p = plot(temperatures, pressures_midpoints, yflip = true, label = "Temperature profile", dpi = 300) #, yaxis=:log)
ylabel!("Pressure (hPa)")
xlabel!("Temperature (K)")

p2 = plot(specific_humidity, pressures_midpoints, color = "red", label = "Specific humidity profile")
xlabel!("Specific humidity (g water/kg air)")

p3 = plot(p, p2, layout = (1,2), yflip = true, title = "2019-06-01 06Z")
savefig(p3, "Figures/profiles.png")

"/Users/jamesyoon/Documents/Radiative_Transfer/Figures/profiles.png"

The MERRA-2 datasets provide temperature, pressure, and specific humidity fields. We now need to import the parameters required for the vSmartMOM CoreRT model. These are imported in a yaml file, but we can make changes to the yaml file once we import them.

### We can finally run the model!

First, we should verify that the Planck function works!

In [4]:
function planck_spectrum_wn_i(T::Real, ν_grid)
    c1 = 1.1910427 * 10^(-5)    # mW/m²-sr-cm⁻¹
    c2 = 1.4387752              # K⋅cm

    # L(ν, T) = c1⋅ν³/(exp(c2⋅ν/T) - 1)
    radiance = c1 .* (ν_grid.^3) ./ (exp.(c2 * ν_grid / T) .- 1) ./ 1000 # Returns in W/m²-sr-cm⁻¹

    return radiance
end

planck_spectrum_wn_i (generic function with 1 method)

In [48]:
parameters = vSmartMOM.CoreRT.parameters_from_yaml("no_absorption_check.yaml");

parameters.T = temperatures;
parameters.p = pressures;
parameters.q = specific_humidity;

model = vSmartMOM.CoreRT.model_from_parameters(parameters); # Create model from the YAML file
no_absorption = vSmartMOM.CoreRT.rt_run(model); # Run the model

FT_dual = Float64
Finished initializing arrays
Fourier Moment: 0/2


┌ Info: Processing on: CPU()
│ With FT: Float64
│ Source Function Integration: true
│ Dimensions: (11, 11, 1991)
└ @ Main.vSmartMOM.CoreRT /Users/jamesyoon/Documents/Radiative_Transfer/vSmartMOM_ISOP.jl/src/CoreRT/rt_run.jl:108
Looping over layers ... 100%|████████████████████████████| Time: 0:00:02


Fourier Moment: 1/2


Looping over layers ... 100%|████████████████████████████| Time: 0:00:01


Fourier Moment: 2/2


┌ Info: Radiance Δ range in %: from 0.0 to 0.0 for m=1
└ @ Main.vSmartMOM.CoreRT /Users/jamesyoon/Documents/Radiative_Transfer/vSmartMOM_ISOP.jl/src/CoreRT/tools/postprocessing_vza.jl:63
Looping over layers ... 100%|████████████████████████████| Time: 0:00:01


───────────────────────────────────────────────────────────────────────────────────────
                                              Time                    Allocations      
                                     ───────────────────────   ────────────────────────
          Tot / % measured:               58.2s /  10.8%           10.9GiB /  99.9%    

Section                      ncalls     time    %tot     avg     alloc    %tot      avg
───────────────────────────────────────────────────────────────────────────────────────
RT Kernel                       216    6.03s   95.5%  27.9ms   9.17GiB   84.0%  43.5MiB
  interaction                   213    4.44s   70.4%  20.9ms   9.05GiB   82.9%  43.5MiB
    interaction inv2            213    1.32s   20.8%  6.18ms   2.51GiB   23.0%  12.1MiB
    interaction inv1 bla        213    1.08s   17.2%  5.08ms   2.51GiB   23.0%  12.1MiB
  elemental                     216    1.54s   24.4%  7.12ms   19.1MiB    0.2%  90.4KiB
  doubling                     

┌ Info: Radiance Δ range in %: from 0.0 to 0.0 for m=2
└ @ Main.vSmartMOM.CoreRT /Users/jamesyoon/Documents/Radiative_Transfer/vSmartMOM_ISOP.jl/src/CoreRT/tools/postprocessing_vza.jl:63


In [73]:
nu = 10:1:2000

planck_function_298 = planck_spectrum_wn_i(298, nu)
planck_function_273 = planck_spectrum_wn_i(273, nu)
planck_function_210 = planck_spectrum_wn_i(210, nu)

p1 = plot(nu, no_absorption[1][1,1,:], lw = 3, label = "MERRA2 Profile, VZA = 0", color = "black", dpi = 300)
plot!(nu, planck_function_210, lw = 1.5, label = "Planck Function (210 K), VZA = 0", dpi = 300)
plot!(nu, planck_function_273, lw = 1.5, label = "Planck Function (273 K), VZA = 0", dpi = 300)
plot!(nu, planck_function_298, lw = 1.5, label = "Planck Function (298 K), VZA = 0", dpi = 300)
xlabel!("Wavenumber (cm-¹)")
ylabel!("Radiance (W/m²-sr-cm-¹)")
title!("No Absorption Nor Scattering, MERRA2 Profile")

savefig(p1, "Figures/no_absorption.png");

## Add in CO2 and O2 absorption

In [ ]:
parameters = vSmartMOM.CoreRT.parameters_from_yaml("yaml_files/CO2_O2_absorption_check.yaml");

parameters.T = temperatures;
parameters.p = pressures;
parameters.q = specific_humidity;

model = vSmartMOM.CoreRT.model_from_parameters(parameters); # Create model from the YAML file
CO2_O2_absorption = vSmartMOM.CoreRT.rt_run(model); # Run the model

┌ Info: Warning, make sure that the VMR is interpolated correctly! Right now, it might be tricky
└ @ Main.vSmartMOM.CoreRT /Users/jamesyoon/Documents/Radiative_Transfer/vSmartMOM_ISOP.jl/src/CoreRT/tools/atmo_prof.jl:79


(params.absorption_params.molecules[i_band])[molec_i] = "CO2"
Computing profile for CO2 with vmr [0.000385, 0.0003850001145941607, 0.00038500025384879586, 0.0003850004219538966, 0.00038500063276675423, 0.0003850009038702479, 0.000385001258001536, 0.0003850017206489304, 0.00038500232154135714, 0.0003850030974095716, 0.00038500409331857524, 0.00038500536414239197, 0.0003850069761741751, 0.00038500900888806304, 0.00038501156012778464, 0.0003850147473088956, 0.00038501870742051154, 0.00038502360124806107, 0.0003850296159776502, 0.0003850369679764387, 0.0003850459051605253, 0.000385056709469902, 0.0003850696986940517, 0.00038508522766744486, 0.0003851036892370286, 0.0003851256512297746, 0.00038515180192201363, 0.0003851828392742149, 0.00038521955635660994, 0.0003852628507344564, 0.0003853137345804603, 0.00038537334173067056, 0.0003854431780424102, 0.0003855250219831911, 0.00038562073333894034, 0.00038573242175436576, 0.0003858624750428068, 0.0003860146818191252, 0.0003861937480754068, 0.000

Progress: 100%|█████████████████████████████████████████| Time: 0:00:32


(params.absorption_params.molecules[i_band])[molec_i] = "O2"
Computing profile for O2 with vmr 0.21 for band #1
FT_dual = Float64
Finished initializing arrays
Fourier Moment: 0/2


┌ Info: Processing on: CPU()
│ With FT: Float64
│ Source Function Integration: true
│ Dimensions: (11, 11, 1991)
└ @ Main.vSmartMOM.CoreRT /Users/jamesyoon/Documents/Radiative_Transfer/vSmartMOM_ISOP.jl/src/CoreRT/rt_run.jl:108
Looping over layers ... 100%|████████████████████████████| Time: 0:00:01


...

Looping over layers ... 100%|████████████████████████████| Time: 0:00:01


Fourier Moment: 2/2


┌ Info: Radiance Δ range in %: from 0.0 to 0.0 for m=1
└ @ Main.vSmartMOM.CoreRT /Users/jamesyoon/Documents/Radiative_Transfer/vSmartMOM_ISOP.jl/src/CoreRT/tools/postprocessing_vza.jl:63
Looping over layers ... 100%|████████████████████████████| Time: 0:00:01


───────────────────────────────────────────────────────────────────────────────────────
                                              Time                    Allocations      
                                     ───────────────────────   ────────────────────────
          Tot / % measured:                527s /   7.4%           87.8GiB /  98.2%    

Section                      ncalls     time    %tot     avg     alloc    %tot      avg
───────────────────────────────────────────────────────────────────────────────────────
Absorption Coeff                  2    33.4s   86.0%   16.7s   74.9GiB   86.9%  37.5GiB
RT Kernel                       216    4.85s   12.5%  22.5ms   9.59GiB   11.1%  45.5MiB
  interaction                   213    3.55s    9.1%  16.7ms   9.49GiB   11.0%  45.6MiB
    interaction inv1 bla        213    1.02s    2.6%  4.78ms   2.73GiB    3.2%  13.1MiB
    interaction inv2            213    813ms    2.1%  3.82ms   2.73GiB    3.2%  13.1MiB
  elemental                    

┌ Info: Radiance Δ range in %: from 0.0 to 0.0 for m=2
└ @ Main.vSmartMOM.CoreRT /Users/jamesyoon/Documents/Radiative_Transfer/vSmartMOM_ISOP.jl/src/CoreRT/tools/postprocessing_vza.jl:63


In [81]:
nu = 10:1:2000

planck_function_298 = planck_spectrum_wn_i(298, nu)
planck_function_273 = planck_spectrum_wn_i(273, nu)
planck_function_210 = planck_spectrum_wn_i(210, nu)

p1 = plot(nu, CO2_O2_absorption[1][1,1,:], lw = 1, label = "MERRA2 Profile, VZA = 0", color = "black", dpi = 300)
plot!(nu, planck_function_210, lw = 1.5, label = "Planck Function (210 K), VZA = 0", dpi = 300)
plot!(nu, planck_function_273, lw = 1.5, label = "Planck Function (273 K), VZA = 0", dpi = 300)
plot!(nu, planck_function_298, lw = 1.5, label = "Planck Function (298 K), VZA = 0", dpi = 300)
xlabel!("Wavenumber (cm-¹)")
ylabel!("Radiance (W/m²-sr-cm-¹)")
title!("CO2 and O2 Absorption, MERRA2 Profile")

savefig(p1, "Figures/CO2_O2_absorption.png");

### Add ISOP and H2O Absorption

In [5]:
parameters = vSmartMOM.CoreRT.parameters_from_yaml("yaml_files/CO2_O2_ISOP_H2O_absorption_check.yaml");

parameters.T = temperatures;
parameters.p = pressures;
parameters.q = specific_humidity;
parameters.absorption_params.vmr["H2O"] = specific_humidity ./ 1000 * 28.96/18.02; # convert to kg/kg from g/kg, then mol/mol
parameters.absorption_params.vmr["ISOP"] = specific_humidity ./ 1000 .* 1e-7;

model = vSmartMOM.CoreRT.model_from_parameters(parameters); # Create model from the YAML file
CO2_O2_ISOP_H2O_absorption = vSmartMOM.CoreRT.rt_run(model); # Run the model

┌ Info: Warning, make sure that the VMR is interpolated correctly! Right now, it might be tricky
└ @ Main.vSmartMOM.CoreRT /Users/jamesyoon/Documents/Radiative_Transfer/vSmartMOM_ISOP.jl/src/CoreRT/tools/atmo_prof.jl:79


(params.absorption_params.molecules[i_band])[molec_i] = "CO2"
Computing profile for CO2 with vmr [0.000385, 0.0003850001145941607, 0.00038500025384879586, 0.0003850004219538966, 0.00038500063276675423, 0.0003850009038702479, 0.000385001258001536, 0.0003850017206489304, 0.00038500232154135714, 0.0003850030974095716, 0.00038500409331857524, 0.00038500536414239197, 0.0003850069761741751, 0.00038500900888806304, 0.00038501156012778464, 0.0003850147473088956, 0.00038501870742051154, 0.00038502360124806107, 0.0003850296159776502, 0.0003850369679764387, 0.0003850459051605253, 0.000385056709469902, 0.0003850696986940517, 0.00038508522766744486, 0.0003851036892370286, 0.0003851256512297746, 0.00038515180192201363, 0.0003851828392742149, 0.00038521955635660994, 0.0003852628507344564, 0.0003853137345804603, 0.00038537334173067056, 0.0003854431780424102, 0.0003855250219831911, 0.00038562073333894034, 0.00038573242175436576, 0.0003858624750428068, 0.0003860146818191252, 0.0003861937480754068, 0.000

Progress: 100%|█████████████████████████████████████████| Time: 0:00:32


(params.absorption_params.molecules[i_band])[molec_i] = "O2"
Computing profile for O2 with vmr 0.21 for band #1
(params.absorption_params.molecules[i_band])[molec_i] = "ISOP"
read_hitran_isopreneComputing profile for ISOP with vmr [2.8061060675099726e-13, 3.422330792091088e-13, 3.7959910059726096e-13, 4.1038783820113164e-13, 4.166083272139076e-13, 4.2227966332575304e-13, 4.2465917431400157e-13, 4.2646352085284886e-13, 4.2652359297790095e-13, 4.2673227653722277e-13, 4.2585279516060835e-13, 4.2171050154138353e-13, 4.1605308069847525e-13, 4.074657681485405e-13, 3.979811936005717e-13, 3.8654193303955254e-13, 3.769012437260244e-13, 3.6913961594109423e-13, 3.595829184632748e-13, 3.495272267173277e-13, 3.371887260072981e-13, 3.2509281027159884e-13, 3.100484718743246e-13, 2.9689333587157306e-13, 2.85915575659601e-13, 2.778988346108235e-13, 2.7219120966037733e-13, 2.700427785384818e-13, 2.6839841211767634e-13, 2.650610213095206e-13, 2.581584340077825e-13, 2.489701728336513e-13, 2.36811501963529

Progress: 100%|█████████████████████████████████████████| Time: 0:01:19


...

Fourier Moment: 2/2


┌ Info: Radiance Δ range in %: from 0.0 to 0.0 for m=1
└ @ Main.vSmartMOM.CoreRT /Users/jamesyoon/Documents/Radiative_Transfer/vSmartMOM_ISOP.jl/src/CoreRT/tools/postprocessing_vza.jl:63
Looping over layers ... 100%|████████████████████████████| Time: 0:00:01


───────────────────────────────────────────────────────────────────────────────────────
                                     

┌ Info: Radiance Δ range in %: from 0.0 to 0.0 for m=2
└ @ Main.vSmartMOM.CoreRT /Users/jamesyoon/Documents/Radiative_Transfer/vSmartMOM_ISOP.jl/src/CoreRT/tools/postprocessing_vza.jl:63


         Time                    Allocations      
                                     ───────────────────────   ────────────────────────
          Tot / % measured:                169s /  80.2%            183GiB /  96.5%    

Section                      ncalls     time    %tot     avg     alloc    %tot      avg
───────────────────────────────────────────────────────────────────────────────────────
Absorption Coeff                  4     126s   92.8%   31.5s    164GiB   93.1%  41.1GiB
RT Kernel                       216    7.06s    5.2%  32.7ms   10.0GiB    5.7%  47.6MiB
  interaction                   213    4.59s    3.4%  21.6ms   9.60GiB    5.4%  46.2MiB
    interaction inv1 bla        213    1.23s    0.9%  5.77ms   2.79GiB    1.6%  13.4MiB
    interaction inv2            213    1.08s    0.8%  5.06ms   2.73GiB    1.5%  13.1MiB
  elemental                     216    1.86s    1.4%  8.63ms    137MiB    0.1%   648KiB
  doubling                      216   50.6ms    0.0%   234μs   12.5M

In [6]:
nu = 10:1:2000

planck_function_298 = planck_spectrum_wn_i(298, nu)
planck_function_273 = planck_spectrum_wn_i(273, nu)
planck_function_210 = planck_spectrum_wn_i(210, nu)

p1 = plot(nu, CO2_O2_ISOP_H2O_absorption[1][1,1,:], lw = 1, label = "MERRA2 Profile, VZA = 0", color = "black", dpi = 300)
plot!(nu, planck_function_210, lw = 1.5, label = "Planck Function (210 K), VZA = 0", dpi = 300)
plot!(nu, planck_function_273, lw = 1.5, label = "Planck Function (273 K), VZA = 0", dpi = 300)
plot!(nu, planck_function_298, lw = 1.5, label = "Planck Function (298 K), VZA = 0", dpi = 300)
xlabel!("Wavenumber (cm-¹)")
ylabel!("Radiance (W/m²-sr-cm-¹)")
title!("CO2, O2, H2O, ISOP Absorption, MERRA2 Profile")

savefig(p1, "Figures/CO2_O2_ISOP_H2O_absorption.png");